# Lecture 2 · Notebook 2 — From counts to vectors: TF-IDF, LSA and PPMI
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture02/lecture02_2_from_counts_to_vectors.ipynb)

**Companion to [Chapters 6–7 of the video lecture](https://nlp-course-cuhksz.github.io/lecture02.html#p6-s113)** (slides 110–145). Before word2vec, the way to turn words into
numbers was to **count** where they occur, **weight** the counts, and **compress** the result. In this notebook you will:

1. build a **TF-IDF** search engine by hand for a tiny library (slides 113–114),
2. use **latent semantic analysis (SVD)** to match a query with documents that share none of its words (slides 116–118),
3. build **word–word co-occurrence vectors** from Wikipedia text and compare **cosine vs. dot product** (slides 140–141), and
4. see how **PPMI weighting** and **SVD** turn noisy counts into useful dense word vectors (slides 142–145).

Runs on CPU in about 2 minutes (one download: the first 10 MB of the `text8` Wikipedia corpus).

In [ ]:
import os, math, collections, requests
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.utils.extmath import randomized_svd

## 1 · TF-IDF search by hand
Each document and the query become vectors with one coordinate per vocabulary term (the **vector space model**, slide 113).
Slide 114's weight is $w_{t,d} = (1 + \log_{10} \mathrm{tf}_{t,d}) \cdot \log_{10}(N / \mathrm{df}_t)$: a word counts more if it is
frequent in this document, and less if it appears in many documents. Documents are ranked by cosine similarity.

In [ ]:
library = {
    "AIMA":        "artificial intelligence a modern approach search logic planning and learning",
    "Python AI":   "learn ai by writing python code machine learning projects in python",
    "Deep Python": "deep learning with python neural networks and code examples",
    "AI for all":  "an introduction to ai for everyone what ai can and cannot do",
    "AI history":  "a brief history of artificial intelligence from turing to deep learning",
    "Joy":         "the joy of cooking recipes for bread soup and cake",
    "Py Cookbook": "python cookbook recipes for data processing code",
    "Greens":      "chinese cooking recipes rice garlic and green vegetables",
}
names = list(library)
docs = [library[n].split() for n in names]
vocab = sorted({w for d in docs for w in d})
w2i = {w: i for i, w in enumerate(vocab)}
N = len(docs)
df = collections.Counter(w for d in docs for w in set(d))
idf = np.array([math.log10(N / df[w]) for w in vocab])

def tfidf(words):
    v = np.zeros(len(vocab))
    for w, tf in collections.Counter(words).items():
        if w in w2i:
            v[w2i[w]] = (1 + math.log10(tf)) * idf[w2i[w]]
    return v

D = np.array([tfidf(d) for d in docs])
def cos(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return 0.0 if na == 0 or nb == 0 else float(a @ b / (na * nb))   # a zero vector has no direction (slide 141)

for w in ["and", "ai", "python", "code", "recipes", "turing"]:
    print(f"idf({w!r:10}) = log10({N}/{df[w]}) = {idf[w2i[w]]:.2f}")

In [ ]:
def search(query, k=4):
    q = tfidf(query.split())
    scores = sorted(((cos(q, D[i]), names[i]) for i in range(N)), reverse=True)
    print(f"{query!r}:", ", ".join(f"{n} {s:.2f}" for s, n in scores[:k]))

search("learn ai by writing python code")
search("python recipes")
search("artificial intelligence")

The last query shows the **semantic gap** (slides 111 and 115): *AI for all* and *Python AI* are about artificial intelligence but
score zero, because they say "ai", not "artificial intelligence". Exact term matching cannot know these are related.

## 2 · Latent semantic analysis
LSA factorizes the term–document matrix, $X = U \Sigma V^\top$, and keeps only the $k$ largest singular values (slide 117). Terms that
occur in similar documents end up close together even if they never co-occur. We use the classic 9-title example from the original LSA
paper (Deerwester et al., 1990): five titles about human–computer interaction (c1–c5) and four about graph theory (m1–m4).

In [ ]:
titles = {
    "c1": "Human machine interface for ABC computer applications",
    "c2": "A survey of user opinion of computer system response time",
    "c3": "The EPS user interface management system",
    "c4": "System and human system engineering testing of EPS",
    "c5": "Relation of user perceived response time to error measurement",
    "m1": "The generation of random binary ordered trees",
    "m2": "The intersection graph of paths in trees",
    "m3": "Graph minors IV widths of trees and well quasi ordering",
    "m4": "Graph minors a survey",
}
terms = ["human", "interface", "computer", "user", "system", "response", "time", "eps", "survey", "trees", "graph", "minors"]
X = np.array([[t.lower().split().count(term) for t in titles.values()] for term in terms], dtype=float)   # 12 terms x 9 docs
U, S, Vt = np.linalg.svd(X, full_matrices=False)
k = 2
Uk, Sk, Vk = U[:, :k], S[:k], Vt[:k].T          # Vk: one row per document
print("singular values:", S.round(2))

def fold_in(words):                               # query -> same space as the rows of Vk:  q_k = Sigma_k^-1 U_k^T q
    q = np.array([words.count(t) for t in terms], dtype=float)
    return q, (Uk.T @ q) / Sk

q, qk = fold_in("human computer interaction".split())
print(f"\n{'doc':4}{'raw cosine':>12}{'LSA cosine':>12}   title")
for j, (d, t) in enumerate(titles.items()):
    print(f"{d:4}{cos(q, X[:, j]):12.2f}{cos(qk, Vk[j]):12.2f}   {t}")

With raw counts, c3 and c5 score 0 — they share no word with the query. In the 2-dimensional LSA space all five HCI titles land close
to the query (cosine 0.85 or more), while the graph-theory titles score below zero. Term vectors behave the same way: *human* and *user*
never appear in the same title, yet LSA puts them close together.

In [ ]:
T = Uk * Sk                                      # term vectors in the LSA space
for a, b in [("human", "user"), ("human", "eps"), ("user", "trees"), ("graph", "trees")]:
    i, j = terms.index(a), terms.index(b)
    print(f"{a:6} ~ {b:6}  raw cosine {cos(X[i], X[j]):5.2f}   LSA cosine {cos(T[i], T[j]):5.2f}")

plt.figure(figsize=(6, 5))
plt.scatter(T[:, 0], T[:, 1], c="tab:blue")
for i, t in enumerate(terms): plt.annotate(t, T[i], color="tab:blue")
Dk = Vk * Sk
plt.scatter(Dk[:, 0], Dk[:, 1], c="tab:red", marker="s")
for j, d in enumerate(titles): plt.annotate(d, Dk[j], color="tab:red")
plt.axhline(0, c="gray", lw=.5); plt.axvline(0, c="gray", lw=.5); plt.title("Terms (blue) and titles (red) in 2-D LSA space")
plt.show()

## 3 · Words as co-occurrence vectors
Now represent **words** instead of documents (slide 140): for each word, count the words within ±4 positions. We use the first 10 MB of
`text8`, a cleaned dump of English Wikipedia (lower-case, no punctuation), and keep the 10,000 most frequent words.

In [ ]:
URL = "https://huggingface.co/datasets/ardMLX/text8/resolve/main/text8"     # a copy of Matt Mahoney's text8 (100 MB)
if not os.path.exists("text8_10MB.txt"):                                       # download only the first 10 MB, once
    r = requests.get(URL, headers={"Range": "bytes=0-9999999"}, timeout=300)
    open("text8_10MB.txt", "w").write(r.text)
words = open("text8_10MB.txt").read().split()[:-1]
cnt = collections.Counter(words)
V = 10_000
vocab = [w for w, _ in cnt.most_common(V)]
w2i = {w: i for i, w in enumerate(vocab)}
ids = np.array([w2i.get(w, -1) for w in words])
print(f"{len(words):,} tokens, {len(cnt):,} types; keeping the top {V:,}")

WINDOW = 4
rows, cols = [], []
for off in range(1, WINDOW + 1):
    a, b = ids[:-off], ids[off:]
    m = (a >= 0) & (b >= 0)
    rows += [a[m], b[m]]; cols += [b[m], a[m]]          # count both directions
rows, cols = np.concatenate(rows), np.concatenate(cols)
C = sp.coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(V, V)).tocsr()   # duplicates are summed
print(f"co-occurrence matrix {C.shape}, {C.nnz:,} non-zero cells ({C.nnz / V**2:.1%} dense)")

**Cosine vs. dot product** (slide 141). `[1, 1]` and `[10, 10]` point the same way, so their cosine is 1 while their dot product grows
with length. With raw counts, length mostly measures *frequency*, so the dot product prefers frequent words:

In [ ]:
print("cos([1,1],[10,10]) =", round(cos(np.array([1, 1]), np.array([10, 10])), 6), "  dot =", np.dot([1, 1], [10, 10]))

def neighbors(M, word, k=8, metric="cos"):
    i = w2i[word]
    if sp.issparse(M):
        s = (M @ M[i].T).toarray().ravel()
        norms = np.sqrt(np.asarray(M.multiply(M).sum(1)).ravel())
    else:
        s = M @ M[i]
        norms = np.linalg.norm(M, axis=1)
    if metric == "cos":
        s = s / (norms * norms[i] + 1e-9)
    s[i] = -np.inf
    return [vocab[j] for j in np.argsort(-s)[:k]]

for w in ["king", "coffee"]:
    print(f"\n{w}\n  dot product:", neighbors(C, w, metric="dot"), "\n  cosine     :", neighbors(C, w))

## 4 · Weighting with PPMI
Raw counts are dominated by *the*, *of* and *and*, which appear next to everything (slide 142). **Pointwise mutual information** asks
whether two words co-occur more than chance: $\mathrm{PMI}(w,c) = \log_2 \frac{P(w,c)}{P(w)P(c)}$. **PPMI** clips negative values to 0.
(We raise context counts to the power 0.75 — a common smoothing trick that keeps rare contexts from getting huge PMI values.)

In [ ]:
def ppmi(C, alpha=0.75):
    C = C.tocoo()
    total = C.sum()
    pw = np.asarray(C.sum(1)).ravel() / total
    cc = np.asarray(C.sum(0)).ravel() ** alpha
    pc = cc / cc.sum()
    pmi = np.log2((C.data / total) / (pw[C.row] * pc[C.col]))
    keep = pmi > 0
    return sp.csr_matrix((pmi[keep], (C.row[keep], C.col[keep])), shape=C.shape)

P = ppmi(C)
for w in ["coffee", "king"]:
    i = w2i[w]
    raw_top = [vocab[j] for j in np.argsort(-C[i].toarray().ravel())[:10]]
    ppmi_top = [vocab[j] for j in np.argsort(-P[i].toarray().ravel())[:10]]
    print(f"{w}\n  top contexts by raw count: {raw_top}\n  top contexts by PPMI     : {ppmi_top}")

## 5 · Dense vectors with SVD
The PPMI rows are 10,000-dimensional and sparse. Truncated SVD keeps the 100 strongest directions (slide 145), so words that occur
with *related* contexts — not only identical ones — end up close (slide 144).

In [ ]:
Uw, Sw, _ = randomized_svd(P, n_components=100, n_iter=5, random_state=0)   # fast truncated SVD
W = Uw * np.sqrt(Sw)                          # a common choice: U * Sigma^0.5
W = W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-9)
for w in ["king", "coffee", "france", "red", "three", "computer", "einstein", "running"]:
    print(f"{w:9} raw: {', '.join(neighbors(C, w, 5)):45}  PPMI+SVD: {', '.join(neighbors(W, w, 5))}")

In [ ]:
groups = {"countries": ["france", "germany", "italy", "spain", "china", "japan", "india", "russia"],
          "numbers":   ["two", "three", "four", "five", "six", "seven", "eight", "nine"],
          "colours":   ["red", "blue", "green", "yellow", "black", "white", "orange", "purple"],
          "family":    ["father", "mother", "son", "daughter", "brother", "sister", "wife", "husband"]}
sel = [w for g in groups.values() for w in g]
Z = W[[w2i[w] for w in sel]]
Z = Z - Z.mean(0)
xy = Z @ np.linalg.svd(Z, full_matrices=False)[2][:2].T     # PCA to 2-D
plt.figure(figsize=(7, 6))
for g, ws in groups.items():
    idx = [sel.index(w) for w in ws]
    plt.scatter(xy[idx, 0], xy[idx, 1], label=g)
    for i in idx: plt.annotate(sel[i], xy[i])
plt.legend(); plt.title("PPMI + SVD word vectors (PCA to 2-D)"); plt.show()

Look at the neighbors after PPMI + SVD: *coffee* moves from unrelated words (*respectively*, *between*) to other crops and foods, *red* finds other colours, and *king*
finds regnal numbers (iii, iv), because in Wikipedia kings are usually followed by one. No one told the model what a colour or a number is:
the groups appear because these words keep similar company — Firth's
*"You shall know a word by the company it keeps"* (slide 133), computed with counting and one SVD.

## Exercises
1. Change `WINDOW` to 1 and to 10 and rebuild the vectors. How do the neighbors of `running` and `coffee` change? Small windows tend
   to give words with the same *function*; large windows give words from the same *topic* (slide 133).
2. Set `alpha=1.0` in `ppmi` (no smoothing). Which words get strange neighbors, and why are rare contexts the problem?
3. In section 2, try `k = 3` or `k = 5` for LSA. What happens to the scores of c3 and c5 as `k` grows?
4. Add two book descriptions of your own to `library` and write a query that TF-IDF ranks badly. Would LSA help?

**Next:** [Notebook 3 — word2vec from scratch](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture02/lecture02_3_word2vec_from_scratch.ipynb)